# Factor-ARIMA Market Cap Rank Prediction — All Rebalance Years (2006–2025)

---

## Overview

This notebook replaces the univariate market cap time-series models (notebooks 02 and 03) with a **factor-driven** return prediction approach built on the Fama-French 5-Factor framework. Rather than extrapolating each stock's own market cap trajectory, we:

1. **Forecast macro factors** — fit ARIMA(5, d, 5) on each of the 5 Fama-French factors (Mkt-RF, SMB, HML, RMW, CMA) plus the risk-free rate, forecasting 60 trading days ahead
2. **Estimate stock exposures** — OLS regression of each stock's excess daily returns on the 5 factors over the training window, yielding an intercept α and 5 factor betas β
3. **Predict returns** — combine forecasted factors with estimated betas to produce a day-by-day predicted excess return path for each stock
4. **Predict market caps** — compound predicted daily returns onto the last observed market cap
5. **Rank and evaluate** — same metrics as notebooks 02/03 for direct comparison

---

## Why This Approach?

The AR/ARIMA models in notebooks 02/03 treat each stock in isolation. Market cap changes are, however, largely driven by **common factors** — broad market moves (Mkt-RF), size premium dynamics (SMB), value/growth rotation (HML), profitability (RMW), and investment style (CMA). If these factors have any predictable structure, modelling them directly and projecting through factor loadings should outperform univariate extrapolation.

**Expected result:** modest improvement for stocks with high factor R² (large-cap, diversified); still poor for idiosyncratic small-caps where α dominates. The key insight is that neither approach can predict the outlier movers that actually drive index membership changes — those require cross-sectional signals beyond time-series modelling.

---

## Comparison to Prior Notebooks

| | Notebook 02 (AR10) | Notebook 03 (Auto-ARIMA BIC) | **This notebook** |
|---|---|---|---|
| Stock model | AR(10) on MCAP changes | ARIMA(p,d,q) BIC on MCAP | OLS factor loadings |
| Factor model | None | None | ARIMA(5, d_adf, 5) per factor |
| Return source | Extrapolated MCAP | Extrapolated MCAP | Factor model + intercept |
| d selection | Fixed d=1 | ADF per stock | ADF per factor |
| Forecast horizon | 60 days | 60 days | 60 days |

---

## 1. Setup

Imports and parameters. All paths are relative to `notebooks/predictions/` → `../../data/`. Change `STEPS`, `TRAIN_DAYS`, or factor ARIMA orders here to experiment.

In [1]:
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import spearmanr
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from joblib import Parallel, delayed

sns.set_theme(style='whitegrid', font_scale=1.05)
warnings.filterwarnings('ignore')

# ── Data paths ────────────────────────────────────────────────────────────────
PROCESSED_DIR  = Path('../../data/processed')
RAW_MBR_DIR    = Path('../../data/raw/membership')

FF5_FILE       = PROCESSED_DIR / 'F-F_Research_Data_5_Factors_2x3_daily.csv'
PRICES_PARQUET = PROCESSED_DIR / 'FINALIZED_PRICES.parquet'
MCAPS_PARQUET  = PROCESSED_DIR / 'FINALIZED_MCAPS.parquet'
CALENDAR_FILE  = RAW_MBR_DIR   / 'russell_calendar.xlsx'
CACHE_DIR      = PROCESSED_DIR / 'factor_arima_cache'
CACHE_DIR.mkdir(exist_ok=True)

# ── Factor model parameters ───────────────────────────────────────────────────
FACTOR_COLS = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
ALL_SERIES  = FACTOR_COLS + ['RF']   # 6 ARIMA models per year
MAX_P       = 5      # fixed AR order for factor ARIMA
MAX_Q       = 5      # fixed MA order for factor ARIMA
MAX_D       = 2      # ADF selects d ∈ {0, 1, 2}
ADF_SIG     = 0.05

# ── Study parameters ─────────────────────────────────────────────────────────
STEPS      = 60    # forecast horizon in trading days (~3 calendar months)
TRAIN_DAYS = 315   # training window
MIN_OBS    = 40    # min non-NaN stock observations for OLS
N_JOBS     = -1

print(f'Factor ARIMA order : ARIMA({MAX_P}, d_adf, {MAX_Q})')
print(f'Factors modelled   : {ALL_SERIES}')
print(f'Forecast horizon   : {STEPS} trading days')
print(f'Training window    : up to {TRAIN_DAYS} trading days')

Factor ARIMA order : ARIMA(5, d_adf, 5)
Factors modelled   : ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
Forecast horizon   : 60 trading days
Training window    : up to 315 trading days


---

## 2. Data Loading

Four datasets are loaded here; all come from `data/processed/` or `data/raw/membership/`.

| Dataset | Description | Notes |
|---|---|---|
| **FF5 factors** | Daily Mkt-RF, SMB, HML, RMW, CMA, RF | Divided by 100 (stored as percentages) |
| **Prices** | Daily closing prices for ~11,545 Bloomberg tickers | Returns computed as `pct_change()`, then aligned to FF5 calendar |
| **Market caps** | Daily market caps (USD millions) for same tickers | Last value before rank date = base for MCAP prediction |
| **Russell Calendar** | Official rank dates for 2006–2025 | `t = 0` for training/evaluation split |

**Calendar alignment note:** Bloomberg price data includes US federal holidays (July 4, Christmas, etc.) that are absent from the FF5 dataset. Filtering returns to only FF5 dates removes these and ensures factor regressions are on a consistent calendar — the same fix used in the event study notebook.

In [2]:
# ── FF5 factors ───────────────────────────────────────────────────────────────
FF5 = pd.read_csv(
    FF5_FILE, skiprows=2, header=None,
    names=['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
)
FF5['Date'] = pd.to_datetime(FF5['Date'].astype(str), format='%Y%m%d')
FF5 = FF5.set_index('Date') / 100
print(f'FF5: {FF5.shape[0]:,} days  ({FF5.index.min().date()} → {FF5.index.max().date()})')

# ── Prices → daily returns, aligned to FF5 calendar ──────────────────────────
prices  = pd.read_parquet(PRICES_PARQUET)
returns = prices.pct_change(fill_method=None)
returns = returns[returns.index.isin(FF5.index)]   # drop Bloomberg-only holiday dates
print(f'Returns: {returns.shape[0]:,} dates × {returns.shape[1]:,} tickers')

# ── Market caps ───────────────────────────────────────────────────────────────
mcaps = pd.read_parquet(MCAPS_PARQUET)
print(f'MCaps  : {mcaps.shape[0]:,} dates × {mcaps.shape[1]:,} tickers')

# ── Russell Calendar ──────────────────────────────────────────────────────────
_cal_raw = pd.read_excel(CALENDAR_FILE, sheet_name='Russell Calendar', header=3)
_cal_raw.columns = ['Year', 'Period', 'Rank_Date', 'Announcement_Date',
                    'Effective_Date', 'Effective_Open', 'Notes']
calendar = (
    _cal_raw[['Year', 'Rank_Date', 'Effective_Date']]
    .dropna(subset=['Year'])
    .assign(
        Year           = lambda d: d['Year'].astype(int),
        Rank_Date      = lambda d: pd.to_datetime(d['Rank_Date']),
        Effective_Date = lambda d: pd.to_datetime(d['Effective_Date']),
    )
    .set_index('Year')
)
print(f'Calendar: {len(calendar)} years  ({calendar.index.min()} – {calendar.index.max()})')

FF5: 15,666 days  (1963-07-02 → 2025-09-30)
Returns: 4,967 dates × 11,545 tickers
MCaps  : 5,310 dates × 11,545 tickers
Calendar: 20 years  (2006 – 2025)


---

## 3. Model Definition

### 3.1 ADF-Based Differencing (`select_d`)

Same as notebook 03. Repeatedly applies the Augmented Dickey-Fuller test and differences until the unit root null is rejected (or `MAX_D = 2` is reached). Factors are designed to be stationary — we expect d = 0 for most factors, though RF (which drifts slowly with monetary policy) may get d = 1.

### 3.2 Factor ARIMA (`fit_factor_arima`)

**Fixed orders: ARIMA(5, d_adf, 5).** Unlike the per-stock BIC search in notebook 03, the factor models use fixed p = 5 and q = 5. The rationale:
- There are only 6 factor series (not thousands), so computational cost is negligible
- Factor time-series are longer (up to 315 days in training) and smoother than individual stocks — higher-order models are less likely to overfit
- We care about the 60-day *level* of the factor path, not its day-to-day noise, so richer lag structure is appropriate

### 3.3 Per-Stock OLS + Return Prediction (`estimate_and_predict`)

For each stock in the training window:
1. **Align** stock returns and FF5 factors to common dates
2. **OLS regression**: `excess_return_t = α + β₁·Mkt-RF_t + β₂·SMB_t + β₃·HML_t + β₄·RMW_t + β₅·CMA_t + ε_t`
3. **Forecast**: for each of the 60 forecast horizons h, `predicted_excess_h = α + β·factor_forecast_h`
4. **Total return**: `total_return_h = predicted_excess_h + RF_forecast_h`
5. **Compound**: `predicted_mcap = last_mcap × Π_{h=1}^{60}(1 + total_return_h)`

Daily compounding (rather than simple summation) correctly captures the path-dependency of returns over a 60-day horizon.

In [ ]:
def select_d(series, max_d=MAX_D, significance=ADF_SIG):
    """ADF unit root test — differences until stationary or max_d reached."""
    s = np.asarray(series.dropna(), dtype=float)
    for d in range(max_d + 1):
        if len(s) < 20:
            return d
        if s.max() == s.min():   # constant series — skip ADF, return current d
            return d
        pval = adfuller(s, autolag='AIC')[1]
        if pval < significance:
            return d
        s = np.diff(s)
    return max_d


def fit_factor_arima(series):
    """
    Fit ARIMA(MAX_P, d_adf, MAX_Q) on a factor series and return
    (forecast pd.Series of length STEPS, d_chosen).
    Returns (None, d) on fitting failure.
    """
    s = series.dropna()
    d = select_d(s)
    try:
        res      = ARIMA(s, order=(MAX_P, d, MAX_Q)).fit()
        forecast = res.get_forecast(steps=STEPS).predicted_mean
        # Ensure the forecast index is integer-based 0..STEPS-1 for easy indexing
        forecast = forecast.reset_index(drop=True)
        return forecast, d
    except Exception:
        return None, d


def estimate_and_predict(ticker, ret_series, ff5_train,
                         factor_forecasts, rf_forecast, last_mcap):
    """
    OLS factor loading estimation + 60-day MCAP prediction for one stock.

    Returns
    -------
    (ticker, predicted_mcap, alpha, betas_dict, r_squared)
    predicted_mcap is None if OLS fails or insufficient data.
    """
    if last_mcap is None or np.isnan(last_mcap) or last_mcap <= 0:
        return (ticker, None, None, None, None)

    # ── Align stock returns to FF5 dates in training window ───────────────────
    common = ret_series.dropna().index.intersection(ff5_train.index)
    if len(common) < MIN_OBS:
        return (ticker, None, None, None, None)

    excess_ret = ret_series.loc[common] - ff5_train.loc[common, 'RF']
    X          = sm.add_constant(ff5_train.loc[common, FACTOR_COLS])

    # ── OLS regression ────────────────────────────────────────────────────────
    try:
        ols    = sm.OLS(excess_ret, X).fit()
        alpha  = ols.params['const']
        betas  = ols.params[FACTOR_COLS].to_dict()
        r2     = ols.rsquared
    except Exception:
        return (ticker, None, None, None, None)

    # ── Predict daily total return for each forecast horizon ──────────────────
    daily_returns = []
    for h in range(STEPS):
        exc = alpha + sum(betas[f] * float(factor_forecasts[f].iloc[h])
                          for f in FACTOR_COLS)
        rf  = float(rf_forecast.iloc[h])
        daily_returns.append(exc + rf)

    # ── Compound returns → predicted MCAP ─────────────────────────────────────
    cumulative_return = np.prod([1 + r for r in daily_returns]) - 1
    predicted_mcap    = last_mcap * (1 + cumulative_return)

    return (ticker, predicted_mcap, alpha, betas, r2)


RUSSELL_BANDS = {
    'Russell 1000':     (1,    1000),
    'Russell 2000':     (1001, 3000),
    'Russell 3000':     (1,    3000),
    'Russell Microcap': (2001, 4000),
}

def in_band(rank_col, lo, hi):
    return (rank_col >= lo) & (rank_col <= hi)

---

## 4. Main Loop — Run Study for Every Rebalance Year

For each year:

1. **Slice training data** — up to 315 trading days of FF5 factors, stock returns, and market caps ending on the rank date
2. **Fit factor ARIMAs** — one ARIMA(5, d_adf, 5) per factor and RF (6 models total); produce 60-day forecast paths
3. **Parallel OLS + prediction** — for each eligible stock: estimate factor loadings, forecast cumulative return, compute predicted MCAP
4. **Rank and evaluate** — predicted vs actual MCAP at the 60-day target date
5. **Store metadata** — factor ARIMA orders (d values), OLS R² distribution, betas per stock
6. **Cache to disk** — resume safely after interruption

Factor ARIMA fits only take seconds (6 models). The bottleneck is the parallel OLS across ~3,500 stocks per year, which runs in a few minutes with all CPU cores.

In [4]:
year_results      = {}
factor_forecasts_store = {}   # year → dict of factor → forecast Series
factor_d_store    = {}        # year → dict of factor → d chosen
ols_meta_store    = {}        # year → DataFrame of (ticker, alpha, betas..., r2)

for year, row in calendar.iterrows():
    rank_date  = row['Rank_Date']
    cache_file = CACHE_DIR / f'year_{year}.pkl'

    # ── Load from cache if available ──────────────────────────────────────────
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            cached = pickle.load(f)
        year_results[year]           = cached['cmp']
        factor_forecasts_store[year] = cached['factor_forecasts']
        factor_d_store[year]         = cached['factor_d']
        ols_meta_store[year]         = cached['ols_meta']
        rho, _ = spearmanr(cached['cmp']['predicted'], cached['cmp']['actual'])
        mae    = cached['cmp']['rank_error'].mean()
        print(f'[{year}] loaded from cache  n={len(cached["cmp"]):,}  ρ={rho:.3f}  MAE={mae:.0f}')
        continue

    # ── Slice training data ───────────────────────────────────────────────────
    ff5_all   = FF5[FF5.index <= rank_date]
    ff5_train = ff5_all.iloc[-TRAIN_DAYS:]
    ret_all   = returns[returns.index <= rank_date]
    ret_train = ret_all.iloc[-TRAIN_DAYS:]
    mcap_snap = mcaps[mcaps.index <= rank_date].iloc[-1]   # last observed MCAP
    all_after = mcaps[mcaps.index > rank_date]

    if len(all_after) < STEPS:
        print(f'[{year}] insufficient post-rank data — skipping')
        continue

    # ── Fit ARIMA(5, d_adf, 5) on each factor and RF ─────────────────────────
    factor_forecasts = {}
    factor_d         = {}
    any_factor_failed = False
    for col in ALL_SERIES:
        fc, d = fit_factor_arima(ff5_train[col])
        factor_d[col] = d
        if fc is None:
            print(f'[{year}] factor ARIMA failed for {col} — skipping year')
            any_factor_failed = True
            break
        factor_forecasts[col] = fc
    if any_factor_failed:
        continue

    d_str = '  '.join(f'{c}:d={factor_d[c]}' for c in ALL_SERIES)
    print(f'[{year}] factors → {d_str}')

    # ── Eligible stocks ───────────────────────────────────────────────────────
    eligible = [
        tkr for tkr in ret_train.columns
        if tkr in mcap_snap.index
        and ret_train[tkr].dropna().shape[0] >= MIN_OBS
        and not np.isnan(mcap_snap[tkr])
        and mcap_snap[tkr] > 0
    ]

    # ── Parallel OLS + prediction per stock ───────────────────────────────────
    rf_forecast = factor_forecasts['RF']
    fit_results = Parallel(n_jobs=N_JOBS, prefer='threads')(
        delayed(estimate_and_predict)(
            tkr,
            ret_train[tkr],
            ff5_train,
            factor_forecasts,
            rf_forecast,
            float(mcap_snap[tkr])
        )
        for tkr in eligible
    )

    # ── Assemble results ──────────────────────────────────────────────────────
    predictions = {}
    ols_rows    = []
    for tkr, pred, alpha, betas, r2 in fit_results:
        if pred is not None and np.isfinite(pred) and pred > 0:
            predictions[tkr] = pred
        row_d = {'Ticker': tkr, 'alpha': alpha, 'r2': r2}
        if betas:
            row_d.update(betas)
        ols_rows.append(row_d)
    ols_meta = pd.DataFrame(ols_rows).set_index('Ticker')

    # ── Actual MCAP at 60-day target ──────────────────────────────────────────
    actual_at_target = all_after.iloc[STEPS - 1]
    target_date      = all_after.index[STEPS - 1]

    pred_s   = pd.Series(predictions)
    actual_s = actual_at_target.reindex(pred_s.index)
    cmp      = pd.DataFrame({'predicted': pred_s, 'actual': actual_s}).dropna()

    if len(cmp) < 100:
        print(f'[{year}] too few stocks ({len(cmp)}) — skipping')
        continue

    cmp['pred_rank']   = cmp['predicted'].rank(ascending=False)
    cmp['actual_rank'] = cmp['actual'].rank(ascending=False)
    cmp['rank_error']  = (cmp['pred_rank'] - cmp['actual_rank']).abs()

    prior_rank_date = calendar.loc[year - 1, 'Rank_Date'] if (year - 1) in calendar.index else None
    if prior_rank_date is not None:
        prior_snap       = mcaps[mcaps.index <= prior_rank_date].iloc[-1]
        cmp['prev_rank'] = prior_snap.reindex(cmp.index).rank(ascending=False)
    else:
        cmp['prev_rank'] = np.nan

    cmp['year']        = year
    cmp['target_date'] = target_date
    cmp['n_train']     = len(ff5_train)

    # ── Cache ─────────────────────────────────────────────────────────────────
    with open(cache_file, 'wb') as f:
        pickle.dump({
            'cmp':              cmp,
            'factor_forecasts': factor_forecasts,
            'factor_d':         factor_d,
            'ols_meta':         ols_meta,
        }, f)

    year_results[year]           = cmp
    factor_forecasts_store[year] = factor_forecasts
    factor_d_store[year]         = factor_d
    ols_meta_store[year]         = ols_meta

    rho, _ = spearmanr(cmp['predicted'], cmp['actual'])
    mae    = cmp['rank_error'].mean()
    n_pos  = (pred_s > 0).mean()
    print(f'[{year}]  target={target_date.date()}  n={len(cmp):,}  '
          f'ρ={rho:.3f}  MAE={mae:.0f}  positive_preds={n_pos:.1%}')

print(f'\nCompleted {len(year_results)} / {len(calendar)} years.')

[2006] factors → Mkt-RF:d=0  SMB:d=0  HML:d=0  RMW:d=0  CMA:d=0  RF:d=1
[2006]  target=2006-08-23  n=6,184  ρ=0.992  MAE=152  positive_preds=100.0%


ValueError: Invalid input, x is constant

---

## 5. Results

### 5.1 Per-Year Summary Table

Layout matches notebooks 02 and 03 for direct comparison. The additional **Mean R²** column shows the average OLS goodness-of-fit across stocks — a useful diagnostic: low R² (< 0.1) would suggest factor loadings have little explanatory power and the predictions are essentially just α × 60 days.

In [ ]:
rows = []
for year, cmp in year_results.items():
    rho, pval = spearmanr(cmp['predicted'], cmp['actual'])
    mean_r2   = ols_meta_store[year]['r2'].mean() if year in ols_meta_store else np.nan
    rows.append({
        'Year':        year,
        'Rank Date':   calendar.loc[year, 'Rank_Date'].date(),
        'Target Date': cmp['target_date'].iloc[0].date(),
        'Train Days':  cmp['n_train'].iloc[0],
        'N Stocks':    len(cmp),
        'Spearman ρ':  round(rho, 4),
        'p-value':     round(pval, 4),
        'MAE Rank':    round(cmp['rank_error'].mean(), 1),
        'Median Err':  round(cmp['rank_error'].median(), 1),
        'Mean R²':     round(mean_r2, 3),
    })

summary = pd.DataFrame(rows).set_index('Year')
summary

### 5.2 Spearman ρ and MAE Rank by Year

Same layout as notebooks 02/03. Key comparisons to make:

- **High ρ years**: when factors explain most of market returns (low-dispersion markets), the factor model should outperform the univariate ARIMA because it correctly predicts the broad direction of returns for most stocks simultaneously.
- **Low ρ / high MAE years**: crisis years (2008, 2020) where factor forecasts are wrong and idiosyncratic moves dominate — the factor model may actually underperform the pure ARIMA because it introduces factor forecast error on top of structural uncertainty.

In [ ]:
years = summary.index.tolist()
rhos  = summary['Spearman ρ'].tolist()
maes  = summary['MAE Rank'].tolist()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax = axes[0]
ax.bar(years, rhos, color='steelblue', edgecolor='white', zorder=3)
ax.axhline(0,             color='black',  lw=0.8, zorder=2)
ax.axhline(1,             color='green',  lw=0.8, linestyle='--', label='Perfect (ρ = 1)', zorder=2)
ax.axhline(np.mean(rhos), color='orange', lw=1.2, linestyle=':', label=f'Mean ρ = {np.mean(rhos):.3f}', zorder=2)
ax.set_ylabel('Spearman ρ', fontsize=11)
ax.set_title('Factor-ARIMA — Spearman ρ by Year  (60-day horizon)', fontsize=12)
ax.set_ylim(-0.05, 1.08)
ax.legend(fontsize=9)
for x, v in zip(years, rhos):
    ax.text(x, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

ax = axes[1]
ax.bar(years, maes, color='darkorange', edgecolor='white', zorder=3)
ax.axhline(np.mean(maes), color='steelblue', lw=1.2, linestyle=':', label=f'Mean MAE = {np.mean(maes):.0f}', zorder=2)
ax.set_ylabel('Mean Absolute Rank Error', fontsize=11)
ax.set_xlabel('Rebalance Year', fontsize=11)
ax.set_title('Mean Absolute Rank Error by Year', fontsize=12)
ax.legend(fontsize=9)
for x, v in zip(years, maes):
    ax.text(x, v + 1, f'{v:.0f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.show()

### 5.3 Factor Forecast Quality

Before interpreting rank prediction results, we should ask: **how good were the factor forecasts themselves?**

This section plots the ARIMA-forecasted factor path vs the realized factor values for two selected years:
- **2020**: COVID shock year — the rank date is early May 2020 (after the initial crash), and the training window includes the March collapse. Factor forecasts will likely be near zero or slightly recovering; realized values will be volatile but generally trending up post-crash.
- **2024**: a more typical year — moderate volatility, well-behaved factors.

The gap between forecast and realized is the **factor model's irreducible error** — no improvement in OLS loadings or compounding can compensate for this.

In [ ]:
PLOT_YEARS = [y for y in [2020, 2024] if y in year_results]

for plot_year in PLOT_YEARS:
    rank_date   = calendar.loc[plot_year, 'Rank_Date']
    target_date = year_results[plot_year]['target_date'].iloc[0]
    forecasts   = factor_forecasts_store[plot_year]

    # Realized factor values in the forecast window
    realized = FF5[(FF5.index > rank_date) & (FF5.index <= target_date)]
    n_actual = min(len(realized), STEPS)

    plot_factors = ['Mkt-RF', 'SMB', 'HML']
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    for ax, col in zip(axes, plot_factors):
        fc_vals   = forecasts[col].values[:n_actual]
        real_vals = realized[col].values[:n_actual]
        x         = np.arange(n_actual)

        # Cumulative paths (compound daily values)
        cum_forecast = np.cumprod(1 + fc_vals) - 1
        cum_realized = np.cumprod(1 + real_vals) - 1

        ax.plot(x, cum_realized * 100, color='steelblue', lw=1.8, label='Realized')
        ax.plot(x, cum_forecast * 100, color='darkorange', lw=1.5, linestyle='--', label='ARIMA forecast')
        ax.axhline(0, color='gray', lw=0.6)
        ax.set_title(col, fontsize=11)
        ax.set_xlabel('Trading days after rank date')
        ax.set_ylabel('Cumulative factor return (%)')
        ax.legend(fontsize=8)

    plt.suptitle(
        f'Factor Forecast Quality — {plot_year}  '
        f'(rank date: {rank_date.date()}, target: {target_date.date()})',
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

### 5.4 OLS Factor Loading Distribution

**What factor loadings tell us:**
- **β_MktRF**: market beta — a stock with β ≈ 1.5 is 50% more sensitive to market moves than the average. This is the dominant factor for most stocks.
- **β_SMB**: size loading — positive values indicate small-cap behavior (appropriate for R2000 constituents); negative values indicate large-cap-like behaviour.
- **β_HML**: value loading — positive = value stock, negative = growth stock.
- **β_RMW**, **β_CMA**: profitability and investment style — often small for individual stocks.

The two panels show:
1. **Heatmap**: mean β per factor per year — tracks how the cross-sectional average exposure to each factor shifts over time
2. **R² distribution**: how much of each stock's return variance is explained by the 5 factors — a direct measure of how much "factor model" vs "idiosyncratic" the universe is

In [ ]:
# ── Mean beta per factor per year ─────────────────────────────────────────────
mean_betas = pd.DataFrame({
    year: ols_meta_store[year][FACTOR_COLS].mean()
    for year in sorted(ols_meta_store)
})

fig, axes = plt.subplots(1, 2, figsize=(17, 5))

ax = axes[0]
sns.heatmap(
    mean_betas,
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    linewidths=0.4, cbar_kws={'label': 'Mean β'},
    ax=ax
)
ax.set_title('Mean OLS Factor Loading (β) by Factor and Year', fontsize=11)
ax.set_xlabel('Rebalance Year')
ax.set_ylabel('Factor')
ax.tick_params(axis='x', rotation=45)

# ── R² distribution across all stock-years ────────────────────────────────────
ax = axes[1]
all_r2 = pd.concat([
    ols_meta_store[year]['r2'].rename(year)
    for year in sorted(ols_meta_store)
], axis=1).values.flatten()
all_r2 = all_r2[~np.isnan(all_r2)]

ax.hist(all_r2, bins=50, color='steelblue', edgecolor='white', zorder=3)
ax.axvline(np.mean(all_r2), color='darkorange', lw=1.5, linestyle='--',
           label=f'Mean R² = {np.mean(all_r2):.3f}')
ax.axvline(np.median(all_r2), color='green', lw=1.2, linestyle=':',
           label=f'Median R² = {np.median(all_r2):.3f}')
ax.set_xlabel('OLS R² (factor model fit per stock)', fontsize=11)
ax.set_ylabel('Number of Stock-Years', fontsize=11)
ax.set_title('Distribution of Factor Model R² — All Stock-Years Pooled', fontsize=11)
ax.legend(fontsize=9)

plt.suptitle('Factor Loading Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'Mean R² (pooled)   : {np.mean(all_r2):.3f}')
print(f'Median R² (pooled) : {np.median(all_r2):.3f}')
print(f'R² > 0.3           : {(all_r2 > 0.3).mean():.1%} of stock-years')
print(f'R² < 0.05          : {(all_r2 < 0.05).mean():.1%} of stock-years  (effectively no factor signal)')

### 5.5 Russell Index Membership Overlap

For each Russell index and each year: of the stocks that actually ended up in that index at the 60-day target date, what fraction did our predicted market cap ranks also place there?

The factor model's performance relative to notebooks 02/03 should be most visible here in **Russell 1000** (where market-wide factors dominate) and least visible in **Russell Microcap** (where idiosyncratic moves are largest and factor R² is lowest).

In [ ]:
overlap_rows = []
for year, cmp in year_results.items():
    for name, (lo, hi) in RUSSELL_BANDS.items():
        pred_in   = set(cmp[in_band(cmp['pred_rank'],   lo, hi)].index)
        actual_in = set(cmp[in_band(cmp['actual_rank'], lo, hi)].index)
        n_actual  = len(actual_in)
        correct   = len(pred_in & actual_in)
        overlap_rows.append({
            'Year':        year,
            'Index':       name,
            'N_Actual':    n_actual,
            'Overlap_Pct': correct / n_actual * 100 if n_actual > 0 else 0,
        })

overlap_df = pd.DataFrame(overlap_rows)

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=True)
for ax, (name, _) in zip(axes.flatten(), RUSSELL_BANDS.items()):
    sub     = overlap_df[overlap_df['Index'] == name]
    mean_ov = sub['Overlap_Pct'].mean()
    ax.bar(sub['Year'], sub['Overlap_Pct'], color='steelblue', edgecolor='white', zorder=3)
    ax.axhline(100,     color='green',  lw=0.8, linestyle='--', label='Perfect (100%)', zorder=2)
    ax.axhline(mean_ov, color='orange', lw=1.2, linestyle=':',  label=f'Mean = {mean_ov:.1f}%', zorder=2)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Year')
    ax.set_ylabel('Membership Overlap (%)')
    ax.set_ylim(0, 112)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=8)

plt.suptitle(
    'Russell Index Membership Overlap: Predicted vs Actual\n'
    '(Factor-ARIMA, 60-day horizon)',
    fontsize=13
)
plt.tight_layout()
plt.show()

### 5.6 Add/Remove Prediction — Russell 2000

The factor model faces the same structural challenge as the univariate approaches: the stocks that actually cross the R2000 boundary are those with the most *unusual* returns — precisely the ones where the factor model fails. A stock entering R2000 from Microcap has grown significantly relative to the small-cap universe; a stock leaving to R1000 has grown relative to the large-cap boundary. These are by definition outliers from the factor model's prediction.

**However**, the factor model may do slightly better at identifying **removes to Microcap** (stocks shrinking): if SMB loads are negative (stock behaving like large-cap despite being in R2000) and the market factor is forecasted to be flat, the model will correctly predict modest negative returns for these stocks — which is the direction of actual shrinkage.

Prior membership proxy: previous year's rank date market caps.

In [ ]:
rebal_rows = []
for year, cmp in year_results.items():
    if cmp['prev_rank'].isna().all():
        continue
    for name, (lo, hi) in RUSSELL_BANDS.items():
        prev_in   = in_band(cmp['prev_rank'],   lo, hi)
        pred_in   = in_band(cmp['pred_rank'],   lo, hi)
        actual_in = in_band(cmp['actual_rank'], lo, hi)

        actual_adds    = (~prev_in) & actual_in
        actual_removes = prev_in   & (~actual_in)
        pred_adds      = (~prev_in) & pred_in
        pred_removes   = prev_in   & (~pred_in)

        tp_adds    = (pred_adds    & actual_adds).sum()
        tp_removes = (pred_removes & actual_removes).sum()
        n_aa = actual_adds.sum();    n_pa = pred_adds.sum()
        n_ar = actual_removes.sum(); n_pr = pred_removes.sum()

        rebal_rows.append({
            'Year':             year,
            'Index':            name,
            'Actual Adds':      n_aa,
            'Add_Precision':    tp_adds    / n_pa if n_pa > 0 else np.nan,
            'Add_Recall':       tp_adds    / n_aa if n_aa > 0 else np.nan,
            'Actual Removes':   n_ar,
            'Remove_Precision': tp_removes / n_pr if n_pr > 0 else np.nan,
            'Remove_Recall':    tp_removes / n_ar if n_ar > 0 else np.nan,
        })

rebal_df = pd.DataFrame(rebal_rows)
r2k = rebal_df[rebal_df['Index'] == 'Russell 2000'].set_index('Year')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (metric_p, metric_r, label) in zip(axes, [
    ('Add_Precision',    'Add_Recall',    'Adds'),
    ('Remove_Precision', 'Remove_Recall', 'Removes'),
]):
    w = 0.35
    x = np.arange(len(r2k))
    ax.bar(x - w/2, r2k[metric_p] * 100, w, label='Precision', color='steelblue',  alpha=0.85, zorder=3)
    ax.bar(x + w/2, r2k[metric_r] * 100, w, label='Recall',    color='darkorange', alpha=0.85, zorder=3)
    ax.axhline(r2k[metric_p].mean() * 100, color='steelblue',  lw=1.3, linestyle=':',
               label=f'Avg Precision = {r2k[metric_p].mean():.0%}', zorder=2)
    ax.axhline(r2k[metric_r].mean() * 100, color='darkorange', lw=1.3, linestyle=':',
               label=f'Avg Recall = {r2k[metric_r].mean():.0%}', zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels(r2k.index, rotation=45)
    ax.set_ylabel('%', fontsize=11)
    ax.set_ylim(0, 112)
    ax.set_title(f'Russell 2000 {label}: Precision & Recall by Year', fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle(
    'Russell 2000 Membership Change Prediction  (Factor-ARIMA, 60-day horizon)\n'
    'Prior membership proxy = previous year\'s rank date market caps',
    fontsize=12
)
plt.tight_layout()
plt.show()

print('Russell 2000 Add/Remove Summary (mean ± std across years):')
print(r2k[['Add_Precision', 'Add_Recall', 'Remove_Precision', 'Remove_Recall']]
      .agg(['mean', 'std'])
      .applymap(lambda v: f'{v:.1%}'))

### 5.7 Pooled Error Distribution

Aggregating all stock-years gives the full picture of model accuracy across the 20-year study period.

**Key questions to answer from this chart:**
- What fraction of stocks are within ±100 rank positions? (practical threshold for index inclusion decisions)
- Is the pooled ρ higher than notebooks 02/03? (tests whether factor model adds rank-ordering signal)
- Is the MAE lower? (tests whether factor model reduces absolute displacement)

If the factor model's pooled ρ and MAE are similar to notebooks 02/03, it confirms that the bottleneck is **fundamental unpredictability of relative market cap moves**, not the choice of model.

In [ ]:
all_errors = pd.concat([cmp['rank_error'] for cmp in year_results.values()])
all_pred   = pd.concat([cmp['predicted']  for cmp in year_results.values()])
all_actual = pd.concat([cmp['actual']     for cmp in year_results.values()])

pooled_rho, _ = spearmanr(all_pred, all_actual)
pooled_mae    = all_errors.mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
sorted_err = np.sort(all_errors.values)
cdf = np.arange(1, len(sorted_err) + 1) / len(sorted_err) * 100
ax.plot(sorted_err, cdf, color='steelblue', lw=2)
for thresh, ypos in [(50, 12), (100, 20), (250, 30), (500, 40)]:
    pct = (all_errors <= thresh).mean() * 100
    ax.axvline(thresh, color='gray', linestyle='--', lw=0.8)
    ax.text(thresh + 10, ypos, f'{pct:.0f}% ≤ {thresh}', fontsize=8.5, color='dimgray')
ax.set_xlabel('Absolute Rank Error', fontsize=11)
ax.set_ylabel('Cumulative % of Stock-Years', fontsize=11)
ax.set_title(
    f'CDF of Rank Errors — All Years Pooled\n'
    f'n = {len(all_errors):,} stock-years  |  MAE = {pooled_mae:.0f}  |  ρ = {pooled_rho:.4f}',
    fontsize=11
)
ax.set_ylim(0, 100)

ax = axes[1]
rho_vals = [spearmanr(cmp['predicted'], cmp['actual'])[0] for cmp in year_results.values()]
ax.hist(rho_vals, bins=10, color='steelblue', edgecolor='white', zorder=3)
ax.axvline(np.mean(rho_vals), color='darkorange', lw=1.5, linestyle='--',
           label=f'Mean ρ = {np.mean(rho_vals):.3f}')
ax.set_xlabel('Spearman ρ', fontsize=11)
ax.set_ylabel('Number of Years', fontsize=11)
ax.set_title('Distribution of Per-Year Spearman ρ', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Pooled Spearman ρ  : {pooled_rho:.4f}')
print(f'Pooled MAE rank    : {pooled_mae:.1f}')
print(f'Total stock-years  : {len(all_errors):,}')